<a href="https://colab.research.google.com/github/ferragina/MyInformationRetrieval/blob/main/8_IndiciEmbeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **TESTING VECTOR INDEXES on SIFT1M**

In [ ]:
%pip install faiss-cpu numpy requests matplotlib pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 80.3 MB/s eta 0:00:00


In [ ]:
import urllib.request as request
import tarfile
import numpy as np
import os
import faiss
import matplotlib.pyplot as plt
import time
import psutil
import tempfile

# Constants
sift_url = "ftp://ftp.irisa.fr/local/texmex/corpus/sift.tar.gz"
sift_archive = "sift.tar.gz"
sift_dir = "sift"

# The dataset is split into a “base” set of vectors (xb) to be indexed and
# a “query” set (xq) to search for. We also have the ground truth (I_true)
# which tells us the actual nearest neighbors for each query,
# which is crucial for evaluating our index’s accuracy later.
base_file = f"{sift_dir}/sift_base.fvecs"
query_file = f"{sift_dir}/sift_query.fvecs"
gt_file = f"{sift_dir}/sift_groundtruth.ivecs"
len_file = f"{sift_dir}/sift_learn.fvecs"

# Download and extract
if not os.path.exists(sift_archive):
    print("Downloading sift.tar.gz...")
    request.urlretrieve(sift_url, sift_archive)
if not os.path.exists(base_file):
    print("Extracting archive...")
    with tarfile.open(sift_archive, "r:gz") as tar:
        tar.extractall()

Extracting archive...


/tmp/ipykernel_2059/533585506.py:27: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall()


In [ ]:
# Loaders
def read_fvecs(filename):
    with open(filename, "rb") as f:
        d = np.fromfile(f, dtype=np.int32, count=1)[0]
        f.seek(0)
        data = np.fromfile(f, dtype=np.float32)
        return data.reshape(-1, d + 1)[:, 1:]
def read_ivecs(filename):
    with open(filename, "rb") as f:
        d = np.fromfile(f, dtype=np.int32, count=1)[0]
        f.seek(0)
        data = np.fromfile(f, dtype=np.int32)
        return data.reshape(-1, d + 1)[:, 1:]

# Load data
xb = read_fvecs(base_file)
xq = read_fvecs(query_file)
I_true = read_ivecs(gt_file)
xt = read_fvecs(len_file)
print("✅ Loaded SIFT1M Dataset")
print(f"Base: {xb.shape}, Query: {xq.shape}, Ground Truth: {I_true.shape} , Learn: {xt.shape}")


✅ Loaded SIFT1M Dataset
Base: (1000000, 128), Query: (10000, 128), Ground Truth: (10000, 100) , Learn: (100000, 128)


# **BRUTE-FORCE SEARCH:: FLAT**

In [ ]:
d = xb.shape[1]

# Create the brute-force index
index_flat = faiss.IndexFlatL2(d)
index_flat.add(xb)
print("Number of vectors in the index:", index_flat.ntotal)

# --- Perform a search on a small subset of queries ---
k = 100  # Number of nearest neighbors to find
num_queries_to_test = len(xq) # Use all #1000
print(f"\nSearching the first {num_queries_to_test} queries with IndexFlatL2...")

start_time = time.time()
D_flat, I_flat = index_flat.search(xq[:num_queries_to_test], k)
end_time = time.time()
print(f"Search completed in {end_time - start_time:.4f} seconds.")

# --- Evaluate the recall ---
# Recall is 1.0 by definition for a brute-force search
#
# The Recall is 0.99 becaue the ground truth is not exact, and there might be
#   slight differences in the nearest neighbors due to floating-point precision issues.
recall_at_1 = (I_flat[:, 0] == I_true[:num_queries_to_test, 0]).sum() / num_queries_to_test
print(f"\nRecall@1 for IndexFlatL2 (subset): {recall_at_1:.4f}")


Number of vectors in the index: 1000000

Searching the first 10000 queries with IndexFlatL2...
Search completed in 63.4724 seconds.

Recall@1 for IndexFlatL2 (subset): 0.9914


# **LOCALITY-SENSITIVE HASHING**

In [ ]:
# LSH is good for fast approximate nearest neighbor search with large datasets
nbits = d * 4  # Controls the number of bits used for hashing; more bits = finer resolution
index_lsh = faiss.IndexLSH(d, nbits)
print("\nBuilding LSH index...")
start_time = time.time()
index_lsh.add(xb)  # Add the same base vectors
end_time = time.time()
print(f"Index built in {end_time - start_time:.4f} seconds.")
print("Number of vectors in the LSH index:", index_lsh.ntotal)

# --- Perform Search with LSH ---
print(f"\nSearching all queries with LSH...")
start_time = time.time()
D_lsh, I_lsh = index_lsh.search(xq, k)
end_time = time.time()
print(f"Search completed in {end_time - start_time:.4f} seconds.")

# --- Evaluate Recall@1 ---
recall_at_1_lsh = (I_lsh[:, 0] == I_true[:, 0]).sum() / len(xq)
print(f"\nRecall@1 for IndexLSH: {recall_at_1_lsh:.4f}")



Building LSH index...
Index built in 4.6897 seconds.
Number of vectors in the LSH index: 1000000

Searching all queries with LSH...
Search completed in 58.7850 seconds.

Recall@1 for IndexLSH: 0.3252


In [ ]:
nbits_list = [2*d, 4*d, 8*d, 32*d, 64*d]
recalls = []
for nbits in nbits_list:
    index_lsh = faiss.IndexLSH(d, nbits)
    index_lsh.add(xb)
    D_lsh, I_lsh = index_lsh.search(xq, k)
    recall_at_1_lsh = (I_lsh[:, 0] == I_true[:, 0]).sum() / len(xq)
    recalls.append(recall_at_1_lsh)

plt.figure(figsize=(8, 5))
plt.plot(nbits_list, recalls, marker='o')
plt.xlabel('nbits')
plt.ylabel('Recall@1')
plt.title('LSH nbits vs Recall@1')
plt.grid(True)
plt.show()

In [ ]:
# Settings
d = 128  # example dimension, set to your actual value
vector_counts = [10_000, 50_000, 100_000, 200_000, 500_000]  # example sizes

In [ ]:
def get_index_size_bytes(index):
    import io
    buf = io.BytesIO()
    faiss.write_index(index, faiss.PyCallbackIOWriter(buf.write))
    return buf.tell()
sizes_lsh_2 = []
sizes_lsh_64 = []
sizes_flat = []
for n in vector_counts:
    xb = np.random.random((n, d)).astype('float32')

    # LSH nbits = d*2
    index_lsh_2 = faiss.IndexLSH(d, d*2)
    index_lsh_2.add(xb)
    size_bytes_2 = get_index_size_bytes(index_lsh_2)
    sizes_lsh_2.append(size_bytes_2 / (1024**3))

    # LSH nbits = d*64
    index_lsh_64 = faiss.IndexLSH(d, d*64)
    index_lsh_64.add(xb)
    size_bytes_64 = get_index_size_bytes(index_lsh_64)
    sizes_lsh_64.append(size_bytes_64 / (1024**3))

    # Flat index
    index_flat = faiss.IndexFlatL2(d)
    index_flat.add(xb)
    size_bytes_flat = get_index_size_bytes(index_flat)
    sizes_flat.append(size_bytes_flat / (1024**3))

plt.figure(figsize=(8, 5))
plt.plot(vector_counts, sizes_lsh_2, marker='o', label=f'LSH nbits={d*2}')
plt.plot(vector_counts, sizes_lsh_64, marker='o', label=f'LSH nbits={d*64}')
plt.plot(vector_counts, sizes_flat, marker='o', label='IndexFlatL2')
plt.xlabel('Number of vectors')
plt.ylabel('Index size (GB)')
plt.title('Index size vs Number of vectors')
plt.legend()

# **HNSW (Hierarchical Navigable Small World)**

In [ ]:
# M is the number of connections for each node. A higher M can improve recall at the cost of memory and build time.
M = 64 # number of connections each vertex will have
index_hnsw = faiss.IndexHNSWFlat(d, M)
print("Building HNSW index...")

# efConstruction controls the quality of the graph construction(how many entry points will be explored when building the index.)
index_hnsw.hnsw.efConstruction = 128
start_time = time.time()
index_hnsw.add(xb)
end_time = time.time()
print(f"Index built in {end_time - start_time:.4f} seconds.")
print("Number of vectors in the HNSW index:", index_hnsw.ntotal)# --- Perform a search ---

# efSearch controls the depth of search. Higher is more accurate and slower.
index_hnsw.hnsw.efSearch = 32
print(f"\nSearching all queries with HNSW (efSearch={index_hnsw.hnsw.efSearch})...")
start_time = time.time()
D_hnsw, I_hnsw = index_hnsw.search(xq, k)
end_time = time.time()
print(f"Search completed in {end_time - start_time:.4f} seconds.")

# --- Evaluate the recall ---
recall_at_1 = (I_hnsw[:, 0] == I_true[:, 0]).sum() / len(xq)
print(f"\nRecall@1 for IndexHNSWFlat: {recall_at_1:.4f}")


In [ ]:
M_values = [16, 32, 64, 128]
efSearch_values = [8, 16, 32, 64, 128]
efConstruction_values = [32, 64, 128, 256]

In [ ]:
recall_results = {}
search_time_results = {}
memory_usage_results = []
for M in M_values:
    recall_results[M] = []
    search_time_results[M] = []

    # Build index for each M
    index_hnsw = faiss.IndexHNSWFlat(d, M)
    index_hnsw.hnsw.efConstruction = 128  # Fixed for recall/search time plots
    index_hnsw.add(xb)

    # Measure memory usage after index is built
    process = psutil.Process()
    memory_usage_results.append(process.memory_info().rss / (1024 ** 2))  # in MB
    for efSearch in efSearch_values:
        index_hnsw.hnsw.efSearch = efSearch
        start_time = time.time()
        D_hnsw, I_hnsw = index_hnsw.search(xq, k)
        end_time = time.time()
        recall_at_1 = (I_hnsw[:, 0] == I_true[:, 0]).sum() / len(xq)
        recall_results[M].append(recall_at_1)
        search_time_results[M].append(end_time - start_time)

# Plot Recall@1 vs efSearch
plt.figure(figsize=(8, 5))
for M in M_values:
    plt.plot(efSearch_values, recall_results[M], label=f'M={M}')
plt.xlabel('efSearch')
plt.ylabel('Recall@1')
plt.title('Recall@1 vs efSearch for different M')
plt.legend()
plt.grid(True)
plt.show()

# Plot Search Time vs efSearch
plt.figure(figsize=(8, 5))
for M in M_values:
    plt.plot(efSearch_values, search_time_results[M], label=f'M={M}')
plt.xlabel('efSearch')
plt.ylabel('Search Time (s)')
plt.title('Search Time vs efSearch for different M')
plt.legend()
plt.grid(True)
plt.show()

# Plot Memory Usage vs M
plt.figure(figsize=(8, 5))
plt.plot(M_values, memory_usage_results, marker='o')
plt.xlabel('M')
plt.ylabel('Index Memory Usage (MB)')
plt.title('Index Memory Usage vs M')
plt.grid(True)
plt.show()

# **Inverted File Index (IndexIVFFlat)**

In [ ]:
nlist = 4096  # A common choice for 1M vectors is sqrt(N), so around 1000. 4*sqrt(N) or 4096 is also common.
quantizer = faiss.IndexFlatL2(d) # how the vectors will be compared in the coarse quantization step
index_ivf = faiss.IndexIVFFlat(quantizer, d, nlist) # Train the index on the 'learn' set
print("Training IndexIVFFlat...")

start_time = time.time()
index_ivf.train(xt)
end_time = time.time()
print(f"Training completed in {end_time - start_time:.4f} seconds.")# Add the full base set to the index
print("Adding vectors to the index...")

index_ivf.add(xb)
print("Number of vectors in the IVF index:", index_ivf.ntotal)# --- Perform a search ---

index_ivf.nprobe = 16 # Higher nprobe increases accuracy and search time
print(f"\nSearching all queries with IndexIVFFlat (nprobe={index_ivf.nprobe})...")

start_time = time.time()
D_ivf, I_ivf = index_ivf.search(xq, k)
end_time = time.time()
print(f"Search completed in {end_time - start_time:.4f} seconds.")# --- Evaluate the recall ---

recall_at_1 = (I_ivf[:, 0] == I_true[:, 0]).sum() / len(xq)
print(f"\nRecall@1 for IndexIVFFlat: {recall_at_1:.4f}")

In [ ]:
nlist_values = [256, 512, 1024, 2048, 4096, 8192]
nprobe_values = [1, 4, 8, 16, 32, 64]

In [ ]:
results = []
memory_usages = []
for nlist in nlist_values:
    quantizer = faiss.IndexFlatL2(d)
    index_ivf = faiss.IndexIVFFlat(quantizer, d, nlist)
    index_ivf.train(xt)
    index_ivf.add(xb)

    # Measure memory usage
    with tempfile.NamedTemporaryFile(delete=False) as tmp:
        faiss.write_index(index_ivf, tmp.name)
        size_bytes = os.path.getsize(tmp.name)
    memory_usages.append(size_bytes / (1024 * 1024))  # MB
    os.remove(tmp.name)

    # Search/recall for each nprobe
    for nprobe in nprobe_values:
        index_ivf.nprobe = nprobe
        start_time = time.time()
        D_ivf, I_ivf = index_ivf.search(xq, k)
        search_time = time.time() - start_time
        recall_at_1 = (I_ivf[:, 0] == I_true[:, 0]).sum() / len(xq)
        results.append({
            'nlist': nlist,
            'nprobe': nprobe,
            'search_time': search_time,
            'recall_at_1': recall_at_1
        })

# Convert results to numpy arrays for plotting
search_times = np.array([r['search_time'] for r in results])
recalls = np.array([r['recall_at_1'] for r in results])
nlists = np.array([r['nlist'] for r in results])
nprobes = np.array([r['nprobe'] for r in results])
nlist_values_unique = sorted(set(r['nlist'] for r in results))

# Plot nprobe vs recall for each nlist
plt.figure(figsize=(10, 5))
for nlist in nlist_values_unique:
    nprobe_vals = [r['nprobe'] for r in results if r['nlist'] == nlist]
    recall_vals = [r['recall_at_1'] for r in results if r['nlist'] == nlist]
    plt.plot(nprobe_vals, recall_vals, marker='o', label=f'nlist={nlist}')
plt.xlabel('nprobe')
plt.ylabel('Recall@1')
plt.title('nprobe vs Recall@1 for different nlist')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# Plot nprobe vs search time for each nlist
plt.figure(figsize=(10, 5))
for nlist in nlist_values_unique:
    nprobe_vals = [r['nprobe'] for r in results if r['nlist'] == nlist]
    search_time_vals = [r['search_time'] for r in results if r['nlist'] == nlist]
    plt.plot(nprobe_vals, search_time_vals, marker='o', label=f'nlist={nlist}')
plt.xlabel('nprobe')
plt.ylabel('Search Time (s)')
plt.title('nprobe vs Search Time for different nlist')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# Plot Memory Usage vs nlist
plt.figure(figsize=(8, 5))
plt.plot(nlist_values, memory_usages, marker='o')
plt.xlabel('nlist')
plt.ylabel('Index Size (MB)')
plt.title('IndexIVFFlat Memory Usage vs nlist')
plt.grid(True)
plt.show()

# **OVERALL COMPARISON**

In [ ]:

# Constants
sift_dir = "sift"
base_file = f"{sift_dir}/sift_base.fvecs"
query_file = f"{sift_dir}/sift_query.fvecs"
gt_file = f"{sift_dir}/sift_groundtruth.ivecs"
learn_file = f"{sift_dir}/sift_learn.fvecs"


In [ ]:
def read_ivecs(filename):
    """Reads ivecs file format."""
    with open(filename, "rb") as f:
        # First 4 bytes are the vector dimension
        d = np.fromfile(f, dtype=np.int32, count=1)[0]
        f.seek(0)
        # Read the whole file as int32
        data = np.fromfile(f, dtype=np.int32)
        # Reshape into (N, d+1) and slice off the first column (dimension)
        return data.reshape(-1, d + 1)[:, 1:].copy()

# Load data
print("--- Loading SIFT1M Dataset ---")
xb = read_fvecs(base_file)
xq = read_fvecs(query_file)
I_true = read_ivecs(gt_file)
xt = read_fvecs(learn_file)
print("✅ Loaded SIFT1M Dataset")
print(f"Base vectors (xb):      {xb.shape}")
print(f"Query vectors (xq):     {xq.shape}")
print(f"Ground truth (I_true):  {I_true.shape}")
print(f"Learn vectors (xt):       {xt.shape}")

# --- Part 2: Index Comparison Framework ---

# --- 1. Data and Ground Truth Setup ---
print("\n--- Initializing Parameters for SIFT1M ---")
d = xb.shape[1]      # Dimension of vectors
nq = xq.shape[0]     # Number of query vectors
# The loaded ground truth contains the top 100 nearest neighbors.
# For Recall@1, we only need the single closest neighbor (the first column).
I_true_1 = I_true[:, :1]
print(f"Using ground truth for Recall@1 evaluation (shape: {I_true_1.shape}).")

# --- 2. Evaluation Framework ---
results = []
k_search = 100  # Number of nearest neighbors to retrieve during tests


def get_memory_usage(index):
    """Measures the memory usage of a Faiss index in MB."""
    # Faiss provides a utility to get the index size directly for some indexes,
    # but writing to a temporary file is a universal method.
    with tempfile.NamedTemporaryFile(delete=False) as tmp:
        faiss.write_index(index, tmp.name)
        size_mb = os.path.getsize(tmp.name) / (1024 * 1024)
    os.remove(tmp.name)
    return size_mb


def evaluate(index, index_name, is_trained=False):
    """Helper function to evaluate build, memory, search, and recall."""
    gc.collect() # Clean up memory before measurements

    # --- Build Time ---
    t0 = time.time()
    if not is_trained: # For indexes like HNSW that don't need separate training
        index.train(xt)
    index.add(xb)
    build_time = time.time() - t0

    # --- Memory Usage ---
    memory_usage = get_memory_usage(index)

    # --- Search Time & Recall ---
    t0 = time.time()
    D, I = index.search(xq, k_search)
    search_time = time.time() - t0
    recall_at_1 = (I[:, 0] == I_true_1[:, 0]).sum() / nq

    # --- Store results ---
    results.append({
        "Index": index_name,
        "Build Time (s)": build_time,
        "Search Time (s)": search_time,
        "QPS": nq / search_time,
        "Recall@1": recall_at_1,
        "Memory (MB)": memory_usage
    })
    print(f"✅ Evaluation complete for {index_name}")

# --- 3. Evaluate Each Index Configuration ---

# ====== Index 1: IndexFlatL2 (The Reference) ======
print("\n--- Evaluating IndexFlatL2 ---")
index_flat = faiss.IndexFlatL2(d)
evaluate(index_flat, "IndexFlatL2 (Brute-Force)", is_trained=True) # is_trained=True skips the .train() step

# ====== Index 2: IndexLSH (Locality-Sensitive Hashing) ======
print("\n--- Evaluating IndexLSH ---")
nbits = d * 8  # A common choice for nbits
index_lsh = faiss.IndexLSH(d, nbits)
evaluate(index_lsh, "IndexLSH", is_trained=True)

# ====== Index 3: IndexHNSWFlat (Hierarchical Navigable Small World) ======
print("\n--- Evaluating IndexHNSWFlat ---")
M = 64
index_hnsw = faiss.IndexHNSWFlat(d, M)
index_hnsw.hnsw.efConstruction = 128

# Build (once)
t0 = time.time()
index_hnsw.add(xb)
hnsw_build_time = time.time() - t0
hnsw_memory = get_memory_usage(index_hnsw)
efSearch_values = [8, 16, 32, 64, 128, 256]
for ef in efSearch_values:
    index_hnsw.hnsw.efSearch = ef
    t0 = time.time()
    D_hnsw, I_hnsw = index_hnsw.search(xq, k_search)
    search_time = time.time() - t0
    recall = (I_hnsw[:, 0] == I_true_1[:, 0]).sum() / nq
    results.append({
        "Index": f"HNSW (efSearch={ef})", "Build Time (s)": hnsw_build_time,
        "Search Time (s)": search_time, "QPS": nq / search_time,
        "Recall@1": recall, "Memory (MB)": hnsw_memory
    })
    print(f"  ✅ HNSW with efSearch = {ef} evaluated.")

# ====== Index 4: IndexIVFFlat (Inverted File) ======
print("\n--- Evaluating IndexIVFFlat ---")
nlist = 4096  # A common choice for SIFT1M (around 4*sqrt(N))
quantizer = faiss.IndexFlatL2(d)
index_ivf = faiss.IndexIVFFlat(quantizer, d, nlist)

# Train and Build (once)
t0 = time.time()
index_ivf.train(xt)
index_ivf.add(xb)
ivf_build_time = time.time() - t0
ivf_memory = get_memory_usage(index_ivf)
nprobe_values = [1, 4, 8, 16, 32, 64]
for nprobe in nprobe_values:
    index_ivf.nprobe = nprobe
    t0 = time.time()
    D_ivf, I_ivf = index_ivf.search(xq, k_search)
    search_time = time.time() - t0
    recall = (I_ivf[:, 0] == I_true_1[:, 0]).sum() / nq
    results.append({
        "Index": f"IVFFlat (nprobe={nprobe})", "Build Time (s)": ivf_build_time,
        "Search Time (s)": search_time, "QPS": nq / search_time,
        "Recall@1": recall, "Memory (MB)": ivf_memory
    })
    print(f"  ✅ IVFFlat with nprobe = {nprobe} evaluated.")

# --- 4. Display Results in a Table ---
df = pd.DataFrame(results).set_index('Index')
print("\n\n--- Comparison Summary on SIFT1M ---")
print(df.to_string(formatters={
    "Build Time (s)": "{:.3f}".format,
    "Search Time (s)": "{:.3f}".format,
    "QPS": "{:,.0f}".format,
    "Recall@1": "{:.4f}".format,
    "Memory (MB)": "{:.2f}".format
}))

# --- 5. Visualization ---
print("\n--- Generating Performance Plot ---")
plt.style.use('seaborn-v0_8-whitegrid')
fig, ax = plt.subplots(figsize=(14, 9))
colors = {'Brute-Force': 'black', 'LSH': 'red', 'HNSW': 'blue', 'IVF': 'green'}
markers = {'Brute-Force': '*', 'LSH': 's', 'HNSW': 'o', 'IVF': '^'}
for index_name, row in df.iterrows():
    base_type = 'HNSW' if 'HNSW' in index_name else \
                'IVF' if 'IVF' in index_name else \
                'LSH' if 'LSH' in index_name else 'Brute-Force'
    ax.scatter(row['Recall@1'], row['QPS'], s=row['Memory (MB)'] * 5,
               c=colors[base_type], marker=markers[base_type],
               alpha=0.7, edgecolors='black')
ax.set_xlabel('Recall@1 (Accuracy)', fontsize=12)
ax.set_ylabel('Queries Per Second (QPS) - Higher is Better', fontsize=12)
ax.set_title('Faiss Index Performance on SIFT1M (Bubble Size ~ Memory)', fontsize=16)
ax.set_yscale('log')
ax.set_xscale('linear')
ax.grid(True, which="both", ls="--")
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='*', color='w', label='IndexFlatL2', markerfacecolor=colors['Brute-Force'], markersize=15),
    Line2D([0], [0], marker='s', color='w', label='IndexLSH', markerfacecolor=colors['LSH'], markersize=12),
    Line2D([0], [0], marker='o', color='w', label='IndexHNSWFlat', markerfacecolor=colors['HNSW'], markersize=12),
    Line2D([0], [0], marker='^', color='w', label='IndexIVFFlat', markerfacecolor=colors['IVF'], markersize=12),
    Line2D([0], [0], marker='o', color='w', label='Bubble Size ~ Memory Usage', markerfacecolor='gray', markersize=12)
]
ax.legend(handles=legend_elements, bbox_to_anchor=(1.04, 1), loc='upper left', title="Index Types")
plt.tight_layout(rect=[0, 0, 0.85, 1])
plt.show()